# Notebook 02 — Feature Engineering
## IPL 2027 Franchise Squad Predictor

**Input:** `data/processed/deliveries_clean.parquet`  
**Output:** `data/processed/batting_features.parquet`, `data/processed/bowling_features.parquet`

| Feature group | What it captures |
|---|---|
| Phase-wise stats | Performance in Powerplay / Middle / Death specifically |
| Exponential form decay | Recent form weighted 3x more than old form |
| Venue performance | Historical avg at each specific ground |
| Opponent matchup | Performance vs each specific team |
| Consistency score | Reliable every match vs boom-or-bust |
| Pressure index | Clutch performance when match is on the line |

**Critical rule:** Every feature uses `shift(1)` — only PAST matches feed into each prediction. No data leakage.

---

## Step 0 — Imports & Paths

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings('ignore')

PROCESSED_DIR = r"E:\My ML Projects\2027 cricket squad prediction\data\processed"

os.makedirs(PROCESSED_DIR, exist_ok=True)

df = pd.read_parquet(PROCESSED_DIR + 'deliveries_clean.parquet')

print(f'Loaded: {df.shape}')
print(f'Seasons: {sorted(df["season"].unique())}')
print(f'Phase values: {df["phase"].unique()}')

Loaded: (155703, 50)
Seasons: [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025]
Phase values: ['Powerplay' 'Middle' 'Death']


## Step 1 — Per-Match Batting Stats (Overall)

In [2]:
bat_match = df.groupby(
    ['match_id', 'season', 'batter', 'batting_team', 'venue', 'batting_team_won']
).agg(
    runs        = ('batter_runs',   'sum'),
    balls_faced = ('legal_ball',    'sum'),
    fours       = ('is_boundary_4', 'sum'),
    sixes       = ('is_boundary_6', 'sum'),
    dot_balls   = ('is_dot_ball',   'sum'),
).reset_index()

bat_match['strike_rate']   = np.where(bat_match['balls_faced'] > 0, bat_match['runs'] / bat_match['balls_faced'] * 100, 0.0)
bat_match['boundary_rate'] = np.where(bat_match['balls_faced'] > 0, (bat_match['fours'] + bat_match['sixes']) / bat_match['balls_faced'], 0.0)
bat_match['did_bat']       = (bat_match['balls_faced'] > 0).astype(int)
bat_match = bat_match.rename(columns={'batter': 'player'})

print(f'Batting match records : {len(bat_match):,}')
print(f'Unique players        : {bat_match["player"].nunique()}')
bat_match.head(3)

Batting match records : 9,886
Unique players        : 439


,match_id,season,player,batting_team,venue,batting_team_won,runs,balls_faced,fours,sixes,dot_balls,strike_rate,boundary_rate,did_bat
0,980901,2016,AM Rahane,Rising Pune Supergiant,Wankhede Stadium,1,66,41,7,3,16,160.975610,0.243902,1
1,980901,2016,AT Rayudu,Mumbai Indians,Wankhede Stadium,0,22,27,2,0,11,81.481481,0.074074,1
2,980901,2016,F du Plessis,Rising Pune Supergiant,Wankhede Stadium,1,34,33,1,3,18,103.030303,0.121212,1


## Step 2 — Phase-wise Batting Stats

Your dataset has phase values: `Powerplay`, `Middle`, `Death` (capitalised exactly like this).
We pivot so each phase becomes its own column.

In [3]:
bat_phase = df.groupby(['match_id', 'batter', 'phase']).agg(
    phase_runs  = ('batter_runs', 'sum'),
    phase_balls = ('legal_ball',  'sum'),
).reset_index()

bat_phase['phase_sr'] = np.where(
    bat_phase['phase_balls'] > 0,
    bat_phase['phase_runs'] / bat_phase['phase_balls'] * 100,
    np.nan   # NaN = didn't face a ball in this phase — NOT zero
)

# Pivot — phase values are 'Death', 'Middle', 'Powerplay'
bat_phase_pivot = bat_phase.pivot_table(
    index   = ['match_id', 'batter'],
    columns = 'phase',
    values  = ['phase_runs', 'phase_balls', 'phase_sr'],
    fill_value = 0
).reset_index()

# Flatten multi-level columns — keeps capitalisation: Death/Middle/Powerplay
bat_phase_pivot.columns = [
    f'{col[0]}_{col[1]}' if col[1] else col[0]
    for col in bat_phase_pivot.columns
]
bat_phase_pivot = bat_phase_pivot.rename(columns={'batter': 'player'})

# Merge onto overall batting
batting = bat_match.merge(bat_phase_pivot, on=['match_id', 'player'], how='left')

print(f'After phase merge: {batting.shape}')
print('Phase columns:', [c for c in batting.columns if 'phase_' in c])

After phase merge: (9886, 23)
Phase columns: ['phase_balls_Death', 'phase_balls_Middle', 'phase_balls_Powerplay', 'phase_runs_Death', 'phase_runs_Middle', 'phase_runs_Powerplay', 'phase_sr_Death', 'phase_sr_Middle', 'phase_sr_Powerplay']


## Step 3 — Per-Match Bowling Stats (Overall + Phase-wise)

In [4]:
df['bowling_team_won'] = 1 - df['batting_team_won']

bowl_match = df.groupby(
    ['match_id', 'season', 'bowler', 'bowling_team', 'venue', 'bowling_team_won']
).agg(
    runs_conceded = ('total_runs',   'sum'),
    balls_bowled  = ('legal_ball',   'sum'),
    wickets       = ('is_wicket',    'sum'),
    dot_balls     = ('is_dot_ball',  'sum'),
    wides         = ('wides',        'sum'),
    no_balls      = ('noballs',      'sum'),
).reset_index()

bowl_match['economy'] = np.where(bowl_match['balls_bowled'] > 0, bowl_match['runs_conceded'] / bowl_match['balls_bowled'] * 6, np.nan)
bowl_match['bowl_sr'] = np.where(bowl_match['wickets'] > 0,      bowl_match['balls_bowled'] / bowl_match['wickets'], np.nan)
bowl_match['dot_pct'] = np.where(bowl_match['balls_bowled'] > 0, bowl_match['dot_balls'] / bowl_match['balls_bowled'], 0.0)
bowl_match['did_bowl'] = (bowl_match['balls_bowled'] > 0).astype(int)
bowl_match = bowl_match.rename(columns={'bowler': 'player', 'bowling_team': 'team', 'bowling_team_won': 'team_won'})

# Phase-wise bowling
bowl_phase = df.groupby(['match_id', 'bowler', 'phase']).agg(
    phase_runs_conceded = ('total_runs', 'sum'),
    phase_balls_bowled  = ('legal_ball', 'sum'),
    phase_wickets       = ('is_wicket',  'sum'),
).reset_index()

bowl_phase['phase_economy'] = np.where(
    bowl_phase['phase_balls_bowled'] > 0,
    bowl_phase['phase_runs_conceded'] / bowl_phase['phase_balls_bowled'] * 6,
    np.nan
)

bowl_phase_pivot = bowl_phase.pivot_table(
    index=['match_id', 'bowler'],
    columns='phase',
    values=['phase_runs_conceded', 'phase_balls_bowled', 'phase_wickets', 'phase_economy'],
    fill_value=0
).reset_index()

bowl_phase_pivot.columns = [
    f'{col[0]}_{col[1]}' if col[1] else col[0]
    for col in bowl_phase_pivot.columns
]
bowl_phase_pivot = bowl_phase_pivot.rename(columns={'bowler': 'player'})

bowling = bowl_match.merge(bowl_phase_pivot, on=['match_id', 'player'], how='left')

print(f'Bowling match records : {len(bowling):,}')
print(f'After phase merge     : {bowling.shape}')

Bowling match records : 7,704
After phase merge     : (7704, 28)


## Step 4 — Add Opponent Column

In [5]:
match_teams = df[['match_id', 'team1', 'team2']].drop_duplicates('match_id')

batting = batting.merge(match_teams, on='match_id', how='left')
batting['opponent'] = np.where(batting['batting_team'] == batting['team1'], batting['team2'], batting['team1'])
batting = batting.drop(columns=['team1', 'team2'])

bowling = bowling.merge(match_teams, on='match_id', how='left')
bowling['opponent'] = np.where(bowling['team'] == bowling['team1'], bowling['team2'], bowling['team1'])
bowling = bowling.drop(columns=['team1', 'team2'])

print('Opponent column added.')

Opponent column added.


## Step 5 — Exponential Form Decay

Recent matches get ~3x the weight of matches from 4 games ago. `shift(1)` prevents any current-match data leaking into its own feature.

In [6]:
def add_ewm_form(df, player_col, metrics, alpha=0.3):
    out = df.sort_values([player_col, 'season', 'match_id']).copy()
    for metric in metrics:
        if metric not in out.columns:
            continue
        out[f'form_{metric}'] = (
            out.groupby(player_col)[metric]
               .transform(lambda x: x.shift(1).ewm(alpha=alpha, min_periods=3).mean())
        )
    return out

batting = add_ewm_form(batting, 'player', ['runs', 'strike_rate', 'fours', 'sixes', 'boundary_rate'])
bowling = add_ewm_form(bowling, 'player', ['economy', 'wickets', 'dot_pct', 'runs_conceded'])

print('Form features added.')
print('Batting:', [c for c in batting.columns if c.startswith('form_')])
print('Bowling:', [c for c in bowling.columns if c.startswith('form_')])

Form features added.
Batting: ['form_runs', 'form_strike_rate', 'form_fours', 'form_sixes', 'form_boundary_rate']
Bowling: ['form_economy', 'form_wickets', 'form_dot_pct', 'form_runs_conceded']


## Step 6 — Venue Performance

In [7]:
def add_venue_avg(df, player_col, metrics):
    out = df.sort_values([player_col, 'venue', 'match_id']).copy()
    for metric in metrics:
        if metric not in out.columns:
            continue
        out[f'venue_avg_{metric}'] = (
            out.groupby([player_col, 'venue'])[metric]
               .transform(lambda x: x.shift(1).expanding().mean())
        )
    return out

batting = add_venue_avg(batting, 'player', ['runs', 'strike_rate'])
bowling = add_venue_avg(bowling, 'player', ['economy', 'wickets'])

print('Venue features added:', [c for c in batting.columns if c.startswith('venue_')])

Venue features added: ['venue_avg_runs', 'venue_avg_strike_rate']


## Step 7 — Opponent Matchup

In [8]:
def add_opponent_avg(df, player_col, metrics):
    out = df.sort_values([player_col, 'opponent', 'match_id']).copy()
    for metric in metrics:
        if metric not in out.columns:
            continue
        out[f'vs_opp_avg_{metric}'] = (
            out.groupby([player_col, 'opponent'])[metric]
               .transform(lambda x: x.shift(1).expanding().mean())
        )
    return out

batting = add_opponent_avg(batting, 'player', ['runs', 'strike_rate'])
bowling = add_opponent_avg(bowling, 'player', ['economy', 'wickets'])

print('Opponent matchup features added:', [c for c in batting.columns if c.startswith('vs_opp_')])

Opponent matchup features added: ['vs_opp_avg_runs', 'vs_opp_avg_strike_rate']


## Step 8 — Consistency Score

In [9]:
def add_consistency(df, player_col, metric, window=10):
    out = df.sort_values([player_col, 'match_id']).copy()
    roll_mean = out.groupby(player_col)[metric].transform(lambda x: x.shift(1).rolling(window, min_periods=3).mean())
    roll_std  = out.groupby(player_col)[metric].transform(lambda x: x.shift(1).rolling(window, min_periods=3).std())
    cv = roll_std / (roll_mean.abs() + 1e-9)
    out[f'consistency_{metric}'] = (1 / (cv + 1e-9)).clip(upper=50)
    return out

batting = add_consistency(batting, 'player', 'runs')
bowling = add_consistency(bowling, 'player', 'wickets')
bowling = add_consistency(bowling, 'player', 'economy')

print('Consistency scores added.')

Consistency scores added.


## Step 9 — Pressure Index (Batting)

Performance while chasing in death overs with required run rate > 10.

In [10]:
# Check which pressure-related columns exist in your dataset
pressure_cols = ['is_chasing', 'is_death_overs', 'required_run_rate']
available = [c for c in pressure_cols if c in df.columns]
print(f'Pressure columns available: {available}')

if len(available) == 3:
    pressure_balls = df[
        (df['is_chasing'] == 1) &
        (df['is_death_overs'] == 1) &
        (df['required_run_rate'] > 10)
    ].copy()

    pressure_stats = pressure_balls.groupby(['match_id', 'batter']).agg(
        pressure_runs  = ('batter_runs', 'sum'),
        pressure_balls = ('legal_ball',  'sum'),
    ).reset_index()

    pressure_stats['pressure_sr'] = np.where(
        pressure_stats['pressure_balls'] > 0,
        pressure_stats['pressure_runs'] / pressure_stats['pressure_balls'] * 100,
        np.nan
    )
    pressure_stats = pressure_stats.rename(columns={'batter': 'player'})

    batting = batting.merge(
        pressure_stats[['match_id', 'player', 'pressure_runs', 'pressure_sr']],
        on=['match_id', 'player'], how='left'
    )
    print(f'Pressure index added. Non-null rows: {batting["pressure_sr"].notna().sum():,}')
else:
    batting['pressure_runs'] = np.nan
    batting['pressure_sr']   = np.nan
    print('Pressure columns not found — added as NaN placeholders. This is fine.')

Pressure columns available: ['is_chasing', 'is_death_overs', 'required_run_rate']
Pressure index added. Non-null rows: 1,517


## Step 10 — Season Recency Weight

In [11]:
MAX_SEASON = 2025
batting['recency_weight'] = 0.85 ** (MAX_SEASON - batting['season'])
bowling['recency_weight'] = 0.85 ** (MAX_SEASON - bowling['season'])

print('Recency weights by season:')
print(batting.groupby('season')['recency_weight'].mean().round(3).to_string())

Recency weights by season:
season
2016    0.232
2017    0.272
2018    0.321
2019    0.377
2020    0.444
2021    0.522
2022    0.614
2023    0.722
2024    0.850
2025    1.000


## Step 11 — Feature Summary

In [12]:
print('=== BATTING FEATURES ===')
print(f'Shape          : {batting.shape}')
print(f'Unique players : {batting["player"].nunique()}')
print()

groups = {
    'Raw match stats'  : ['runs','balls_faced','fours','sixes','strike_rate','boundary_rate'],
    'Phase stats'      : [c for c in batting.columns if 'phase_' in c],
    'Form (EWM)'       : [c for c in batting.columns if c.startswith('form_')],
    'Venue avg'        : [c for c in batting.columns if c.startswith('venue_avg_')],
    'Opponent matchup' : [c for c in batting.columns if c.startswith('vs_opp_')],
    'Consistency'      : [c for c in batting.columns if 'consistency' in c],
    'Pressure'         : ['pressure_runs', 'pressure_sr'],
}

total_features = 0
for group, cols in groups.items():
    existing = [c for c in cols if c in batting.columns]
    total_features += len(existing)
    print(f'  {group:20s} : {len(existing)} features')

print(f'\n  Total feature columns : {total_features}')

print()
print('=== BOWLING FEATURES ===')
print(f'Shape          : {bowling.shape}')
print(f'Unique players : {bowling["player"].nunique()}')

=== BATTING FEATURES ===
Shape          : (9886, 37)
Unique players : 439

  Raw match stats      : 6 features
  Phase stats          : 9 features
  Form (EWM)           : 5 features
  Venue avg            : 2 features
  Opponent matchup     : 2 features
  Consistency          : 1 features
  Pressure             : 2 features

  Total feature columns : 27

=== BOWLING FEATURES ===
Shape          : (7704, 40)
Unique players : 347


## Step 12 — Save Feature Tables

In [13]:
batting.to_parquet(PROCESSED_DIR + 'batting_features.parquet', index=False)
bowling.to_parquet(PROCESSED_DIR + 'bowling_features.parquet', index=False)

print('Saved:')
print(f'  batting_features.parquet — {len(batting):,} rows × {batting.shape[1]} cols')
print(f'  bowling_features.parquet — {len(bowling):,} rows × {bowling.shape[1]} cols')

Saved:
  batting_features.parquet — 9,886 rows × 37 cols
  bowling_features.parquet — 7,704 rows × 40 cols


## Step 13 — Validation Checks

In [14]:
bat = pd.read_parquet(PROCESSED_DIR + 'batting_features.parquet')
bwl = pd.read_parquet(PROCESSED_DIR + 'bowling_features.parquet')

# Row count thresholds calibrated to your dataset:
# 652 matches × ~15 batters = ~9,800 batting rows
# 652 matches × ~12 bowlers = ~7,600 bowling rows
checks = [
    ('batting rows > 9000',           len(bat) > 9000),
    ('bowling rows > 7000',           len(bwl) > 7000),
    ('form_runs exists',              'form_runs' in bat.columns),
    ('form_economy exists',           'form_economy' in bwl.columns),
    ('venue_avg_runs exists',         'venue_avg_runs' in bat.columns),
    ('consistency_runs exists',       'consistency_runs' in bat.columns),
    ('opponent column exists',        'opponent' in bat.columns),
    ('recency_weight exists',         'recency_weight' in bat.columns),
    ('batting_team_won no nulls',     bat['batting_team_won'].isnull().sum() == 0),
    ('bowling team_won no nulls',     bwl['team_won'].isnull().sum() == 0),
    ('phase columns exist (batting)', any('phase_' in c for c in bat.columns)),
    ('phase columns exist (bowling)', any('phase_' in c for c in bwl.columns)),
]

all_pass = True
for name, result in checks:
    icon = '✓' if result else '✗  FAIL'
    print(f'{icon}  {name}')
    if not result:
        all_pass = False

print()
if all_pass:
    print('All checks passed. Proceed to Notebook 03 — Label Creation.')
else:
    print('Fix failing checks before proceeding.')

✓  batting rows > 9000
✓  bowling rows > 7000
✓  form_runs exists
✓  form_economy exists
✓  venue_avg_runs exists
✓  consistency_runs exists
✓  opponent column exists
✓  recency_weight exists
✓  batting_team_won no nulls
✓  bowling team_won no nulls
✓  phase columns exist (batting)
✓  phase columns exist (bowling)

All checks passed. Proceed to Notebook 03 — Label Creation.
